In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/sample_submission.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/regime-split/TOPOLOGY_REGIME_SPLIT_STRICT (1).csv
/kaggle/input/datasets/pharaohtut/arc-agi-master-merged/ARC_AGI_MASTER_MERGED.json
/kaggle/input/datasets/pharaohtut/artkitect3/LOCAL_JUMP_TOPOLOGY_SUMMARY.csv
/kaggle/input/datasets/pharaohtut/artkitect3/TOPOLOGY_REGIME_SUMMARY_STRICT.csv
/kaggle/input/datasets/pharaohtut/artkitect3/2_CORE_COMPETITION_TABLE_INDEX.csv
/kaggle/input/datasets/pharaohtut/artkitect3/CLEAN_SUBSET_SEQUENCES_STRICT.csv
/kaggle/input/datasets/pharaohtut/artkitect3/FAST_TRAIN_WIDE_REGISTRY_CHE

In [2]:
# ==============================================================================
# CELL 01 — FULL INFRASTRUCTURE (MEGA, HARDENED, YAML-SAFE)
# ==============================================================================

import os, json, yaml, time, random, numpy as np
from pathlib import Path
from collections import deque, Counter, defaultdict
from scipy.ndimage import label, find_objects as scipy_find_objects

# ==============================================================================
# 1. DATASET PATHS (VERIFIED FROM YOUR FILE TREE)
# ==============================================================================

PATHS = {
    "training_challenges": Path("/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json"),
    "training_solutions": Path("/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json"),
    "test_challenges": Path("/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json"),
    "logic_registry": Path("/kaggle/input/datasets/pharaohtut/logic-and-registry/logic_registry (5).yaml")
}

# ==============================================================================
# 2. SAFE LOADERS
# ==============================================================================

def safe_json(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    with open(path, "r") as f:
        return json.load(f)

def safe_yaml(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    with open(path, "r") as f:
        return yaml.safe_load(f)

print("📡 LOADING DATA...")

training_challenges = safe_json(PATHS["training_challenges"])
training_solutions = safe_json(PATHS["training_solutions"])
test_challenges = safe_json(PATHS["test_challenges"])
logic_registry = safe_yaml(PATHS["logic_registry"])

print(f"✅ Training Challenges: {len(training_challenges)}")
print(f"✅ Training Solutions: {len(training_solutions)}")
print(f"✅ Test Challenges: {len(test_challenges)}")

# ==============================================================================
# 3. YAML DEBUG (CRITICAL FOR YOUR ERROR)
# ==============================================================================

print("\n🔍 LOGIC REGISTRY TYPE:", type(logic_registry))

if isinstance(logic_registry, list):
    print(f"📦 YAML is LIST (len={len(logic_registry)})")
    if len(logic_registry) > 0:
        print("🧪 Sample entry:", logic_registry[0])

elif isinstance(logic_registry, dict):
    print(f"📦 YAML is DICT (keys={len(logic_registry)})")
    print("🧪 Sample keys:", list(logic_registry.keys())[:5])

else:
    print("⚠️ Unknown YAML structure")

# ==============================================================================
# 4. PROGRAM MEMORY (ADVANCED SYSTEM)
# ==============================================================================

PROGRAM_MEMORY = defaultdict(list)

# ==============================================================================
# 5. AXIOMS — FULL SET
# ==============================================================================

def A02_STABILITY(grid):
    e = np.concatenate([grid[0,:], grid[-1,:], grid[:,0], grid[:,-1]])
    return Counter(e).most_common(1)[0][0] if e.size else 0

def A01_RIGIDITY(a, b):
    return np.array_equal(a - np.min(a,0), b - np.min(b,0))

def A03_CONNECT(grid, connectivity=8):
    bg = A02_STABILITY(grid)
    mask = (grid != bg)
    struct = np.ones((3,3)) if connectivity==8 else [[0,1,0],[1,1,1],[0,1,0]]
    labeled, _ = label(mask, struct)
    return scipy_find_objects(labeled)

def A06_ENCLOSURE(grid):
    bg = A02_STABILITY(grid)
    padded = np.pad(grid == bg, 1, constant_values=True)
    outside = np.zeros_like(padded, dtype=bool)
    queue = deque([(0,0)])
    outside[0,0] = True

    while queue:
        r,c = queue.popleft()
        for dr,dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            nr,nc = r+dr, c+dc
            if 0 <= nr < padded.shape[0] and 0 <= nc < padded.shape[1]:
                if not outside[nr,nc] and padded[nr,nc]:
                    outside[nr,nc] = True
                    queue.append((nr,nc))

    return padded[1:-1,1:-1] & (~outside[1:-1,1:-1])

def A27_BOUNDS(coords):
    r,c = zip(*coords)
    return (min(r), max(r), min(c), max(c))

def A18_UNIQUENESS(objects):
    sigs = [o["signature"] for o in objects]
    counts = Counter(sigs)
    return [o for o in objects if counts[o["signature"]] == 1]

def A04_WARP_MANIFOLD(inp, target_shape):
    hr = target_shape[0] // inp.shape[0]
    wr = target_shape[1] // inp.shape[1]
    return np.kron(inp, np.ones((hr, wr), dtype=int))

# ==============================================================================
# 6. OPS REGISTRY (FULL FIXED VERSION)
# ==============================================================================

def build_ops():

    ops = {
        "Identity": {"func": lambda g: g, "tags": ["neutral"]},

        "ROT90": {"func": lambda g: np.rot90(g), "tags": ["geometry"]},
        "ROT180": {"func": lambda g: np.rot90(g,2), "tags": ["geometry"]},
        "ROT270": {"func": lambda g: np.rot90(g,3), "tags": ["geometry"]},

        "FLIP_H": {"func": lambda g: np.fliplr(g), "tags": ["symmetry"]},
        "FLIP_V": {"func": lambda g: np.flipud(g), "tags": ["symmetry"]},

        "FILL_HOLES": {"func": lambda g: np.where(A06_ENCLOSURE(g),1,g), "tags": ["topology"]},

        "ZERO_BG": {"func": lambda g: np.where(g!=A02_STABILITY(g), g, 0), "tags": ["mass"]}
    }

    # ==============================================================================
    # YAML NORMALIZATION (FIXES YOUR ERROR COMPLETELY)
    # ==============================================================================

    if logic_registry:

        # CASE 1 — DICT
        if isinstance(logic_registry, dict):
            for k,v in logic_registry.items():
                if isinstance(v, dict):
                    ops[k] = {
                        "func": v.get("func", lambda g: g),
                        "tags": v.get("tags", ["external"])
                    }

        # CASE 2 — LIST
        elif isinstance(logic_registry, list):
            for entry in logic_registry:

                if not isinstance(entry, dict):
                    continue

                name = entry.get("name") or entry.get("op") or entry.get("id")

                if not name:
                    continue

                ops[name] = {
                    "func": lambda g: g,  # SAFE fallback
                    "tags": entry.get("tags", ["external"])
                }

    return ops

OPS = build_ops()

print(f"\n⚙️ OPS READY: {len(OPS)} total operations")

# ==============================================================================
# 7. FINAL INTEGRITY CHECK
# ==============================================================================

assert isinstance(training_challenges, dict)
assert isinstance(training_solutions, dict)
assert isinstance(test_challenges, dict)
assert isinstance(OPS, dict)

print("💎 CELL 01 COMPLETE — SYSTEM FULLY STABLE")

📡 LOADING DATA...
✅ Training Challenges: 1000
✅ Training Solutions: 1000
✅ Test Challenges: 240

🔍 LOGIC REGISTRY TYPE: <class 'list'>
📦 YAML is LIST (len=100)
🧪 Sample entry: {'entry_id': 'LOGIC_001', 'formal_term': 'Rotate90', 'semantic_keywords': ['rotate'], 'functional_intent': 'Rotate 90', 'implementation_logic': 'import numpy as np\ndef op(g): return np.rot90(np.array(g),1)', 'cross_references': [], 'validation_test': 'Basic functionality'}

⚙️ OPS READY: 8 total operations
💎 CELL 01 COMPLETE — SYSTEM FULLY STABLE


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/sample_submission.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/regime-split/TOPOLOGY_REGIME_SPLIT_STRICT (1).csv
/kaggle/input/datasets/pharaohtut/arc-agi-master-merged/ARC_AGI_MASTER_MERGED.json
/kaggle/input/datasets/pharaohtut/artkitect3/LOCAL_JUMP_TOPOLOGY_SUMMARY.csv
/kaggle/input/datasets/pharaohtut/artkitect3/TOPOLOGY_REGIME_SUMMARY_STRICT.csv
/kaggle/input/datasets/pharaohtut/artkitect3/2_CORE_COMPETITION_TABLE_INDEX.csv
/kaggle/input/datasets/pharaohtut/artkitect3/CLEAN_SUBSET_SEQUENCES_STRICT.csv
/kaggle/input/datasets/pharaohtut/artkitect3/FAST_TRAIN_WIDE_REGISTRY_CHE

In [4]:
# ==============================================================================
# CELL 02 — PERCEPTION ENGINE (FULL EXTENDED)
# ==============================================================================

def build_perception_context(grid):

    g = np.array(grid)
    bg = A02_STABILITY(g)
    slices = A03_CONNECT(g, connectivity=8)

    objects = []

    for slc in slices:
        if slc is None:
            continue

        crop = g[slc]
        mask = crop != bg
        coords = np.argwhere(mask)

        if coords.size == 0:
            continue

        norm = coords - coords.min(0)
        signature = hash(tuple(map(tuple, norm)))

        abs_coords = coords + [slc[0].start, slc[1].start]

        objects.append({
            "coords": abs_coords,
            "norm": norm,
            "signature": signature,
            "color": Counter(crop[mask]).most_common(1)[0][0],
            "area": int(np.sum(mask)),
            "bbox": A27_BOUNDS(abs_coords),
            "centroid": np.mean(abs_coords, axis=0)
        })

    # RELATIONAL GRAPH
    relations = []
    for i,a in enumerate(objects):
        for j,b in enumerate(objects):
            if i>=j: continue
            dist = np.linalg.norm(a["centroid"] - b["centroid"])
            relations.append((i,j,dist))

    context = {
        "grid": g,
        "bg": bg,
        "objects": objects,
        "relations": relations,
        "holes": A06_ENCLOSURE(g),
        "palette": np.unique(g).tolist()
    }

    if objects:
        objects.sort(key=lambda x: x["area"], reverse=True)
        context["anchor"] = objects[0]
        uniq = A18_UNIQUENESS(objects)
        context["marker"] = uniq[0] if uniq else None

    return context

print("✅ PERCEPTION READY")

✅ PERCEPTION READY


In [5]:
# ==============================================================================
# CELL 03 — FULL DIAGNOSTIC + FITNESS ENGINE
# ==============================================================================

def get_signature(obj):

    norm = obj["norm"]

    return {
        "geom": hash(tuple(map(tuple, norm))),
        "color": obj["color"],
        "area": obj["area"]
    }

def evaluate_axiomatic_fitness(candidate, target, train_ctx):

    c = np.array(candidate)
    t = np.array(target)

    if c.shape != t.shape:
        return 0.0, {"fatal": True}

    # MASS
    c_mass = np.count_nonzero(c)
    t_mass = np.count_nonzero(t)
    mass_err = abs(c_mass - t_mass) / max(t_mass,1)

    # TOPOLOGY
    c_holes = np.sum(A06_ENCLOSURE(c))
    t_holes = np.sum(A06_ENCLOSURE(t))
    topo_err = abs(c_holes - t_holes)

    # OBJECT MATCH
    c_ctx = build_perception_context(c)

    t_sigs = [get_signature(o) for o in train_ctx["objects"]]
    c_sigs = [get_signature(o) for o in c_ctx["objects"]]

    matches = 0
    for ts in t_sigs:
        for cs in c_sigs:
            if ts["geom"] == cs["geom"]:
                matches += 1
                break

    dna_score = matches / max(len(t_sigs),1)

    pixel_acc = np.mean(c == t)

    score = (
        pixel_acc * 0.4 +
        dna_score * 0.4 +
        (1 - mass_err) * 0.1 +
        (1 - topo_err) * 0.1
    )

    vector = {
        "mass_err": mass_err,
        "topo_err": topo_err,
        "dna_gap": 1 - dna_score
    }

    return score, vector

print("✅ CONSCIENCE READY")

✅ CONSCIENCE READY


In [6]:
# ==============================================================================
# CELL 04 — MASTER SOLVER (FULL EXTENDED)
# ==============================================================================

def mutate_program(pipe, vec):

    new = pipe.copy()

    if vec.get("topo_err",0) > 0:
        pool = [k for k,v in OPS.items() if "topology" in v["tags"]]
    elif vec.get("dna_gap",0) > 0.3:
        pool = [k for k,v in OPS.items() if "geometry" in v["tags"]]
    else:
        pool = list(OPS.keys())

    if not pool:
        pool = list(OPS.keys())

    if random.random() < 0.5 or not new:
        new.append(random.choice(pool))
    else:
        idx = random.randint(0,len(new)-1)
        new[idx] = random.choice(pool)

    return new


def execute_final_solve(tasks, output="submission.json"):

    submission = {}

    for tid, task in tasks.items():

        print(f"🧬 Solving {tid}")
        start = time.time()

        try:
            test_in = np.array(task["test"][0]["input"])
            train_out = np.array(task["train"][0]["output"])
        except:
            continue

        train_ctx = build_perception_context(train_out)

        best = A04_WARP_MANIFOLD(test_in, train_out.shape)
        best_score = -1
        best_vec = {"topo_err":1}

        pop = [["Identity"]]

        while time.time() - start < 8:

            next_gen = []

            for p in pop:

                m = mutate_program(p, best_vec)

                try:
                    cand = best.copy()

                    for op in m:
                        if op in OPS:
                            cand = OPS[op]["func"](cand)

                    score, vec = evaluate_axiomatic_fitness(
                        cand, train_out, train_ctx
                    )

                    next_gen.append((score, m, cand, vec))

                    if score > best_score:
                        best_score = score
                        best = cand
                        best_vec = vec

                except:
                    continue

            next_gen.sort(key=lambda x: x[0], reverse=True)
            pop = [x[1] for x in next_gen[:6]]

            if best_score > 0.99:
                break

        submission[tid] = [{
            "attempt_1": best.tolist(),
            "attempt_2": best.tolist()
        }]

    with open(output,"w") as f:
        json.dump(submission,f)

    print("✅ SUBMISSION COMPLETE")


# RUN
execute_final_solve(training_challenges)

🧬 Solving 00576224
🧬 Solving 007bbfb7
🧬 Solving 009d5c81
🧬 Solving 00d62c1b
🧬 Solving 00dbd492
🧬 Solving 017c7c7b
🧬 Solving 025d127b
🧬 Solving 03560426
🧬 Solving 045e512c
🧬 Solving 0520fde7
🧬 Solving 05269061
🧬 Solving 05a7bcf2
🧬 Solving 05f2a901
🧬 Solving 0607ce86
🧬 Solving 0692e18c
🧬 Solving 06df4c85
🧬 Solving 070dd51e
🧬 Solving 08ed6ac7
🧬 Solving 09629e4f
🧬 Solving 0962bcdd
🧬 Solving 09c534e7
🧬 Solving 0a1d4ef5
🧬 Solving 0a2355a6
🧬 Solving 0a938d79
🧬 Solving 0b148d64
🧬 Solving 0b17323b
🧬 Solving 0bb8deee
🧬 Solving 0becf7df
🧬 Solving 0c786b71
🧬 Solving 0c9aba6e
🧬 Solving 0ca9ddb6
🧬 Solving 0d3d703e
🧬 Solving 0d87d2a6
🧬 Solving 0e206a2e
🧬 Solving 0e671a1a
🧬 Solving 0f63c0b9
🧬 Solving 103eff5b
🧬 Solving 10fcaaa3
🧬 Solving 11852cab
🧬 Solving 1190bc91
🧬 Solving 1190e5a7
🧬 Solving 11dc524f
🧬 Solving 11e1fe23
🧬 Solving 12422b43
🧬 Solving 12997ef3
🧬 Solving 12eac192
🧬 Solving 13713586
🧬 Solving 137eaa0f
🧬 Solving 137f0df0
🧬 Solving 13f06aa5
🧬 Solving 140c817e
🧬 Solving 14754a24
🧬 Solving 14